# IADD YOLOv11 - leakage-free (clean) evaluation

IADD contains a few duplicate videos under different Record IDs (e.g. Record416_R ==
Record438_R), so a small fraction of val/test frames have a near-identical twin in
train. This notebook removes those leaked frames (list in `eval_exclude.txt`, found by
perceptual hashing) and re-evaluates run5 on the clean val/test. No retraining needed.

Before running: upload `eval_exclude.txt` to the root of your Drive (My Drive).

## 1. GPU / install / mount

In [ ]:
!nvidia-smi
!pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

## 2. Unzip the dataset

In [ ]:
!rm -rf /content/iadd_subset_v4
!unzip -q "/content/drive/MyDrive/iadd_subset_v4.zip" -d /content

## 3. Remove the leaked (near-duplicate) evaluation frames

Deletes the image + label of every frame listed in eval_exclude.txt from the local
copy only (the Drive zip is untouched).

In [ ]:
import os

DS = '/content/iadd_subset_v4'
lines = [l.strip() for l in open('/content/drive/MyDrive/eval_exclude.txt') if l.strip()]
removed = 0
for entry in lines:
    split, stem = entry.split('/', 1)          # e.g. 'test/Record407_D__..._0'
    for sub, ext in (('images', '.jpg'), ('labels', '.txt')):
        p = f'{DS}/{sub}/{split}/{stem}{ext}'
        if os.path.exists(p):
            os.remove(p); removed += 1
print('removed', removed, 'files (', len(lines), 'frames )')
import glob
for s in ('val', 'test'):
    print(f'  clean {s}: {len(glob.glob(f"{DS}/images/{s}/*.jpg"))} images')

## 4. Clean evaluation of run5 (val + test + pooled)

Honest mAP at default conf, confusion matrix at the val-F1-optimal conf, and the
pooled val+test number - all on the leakage-free sets.

In [ ]:
from ultralytics import YOLO
import numpy as np, os, shutil, glob, yaml

DATA = '/content/iadd_subset_v4/data.yaml'
RUNS = '/content/drive/MyDrive/iadd_runs'
best = YOLO(f'{RUNS}/run5/weights/best.pt')

def clean_eval(split, tag):
    m = best.val(data=DATA, split=split, plots=True, project=RUNS, name=f'run5_clean_{tag}', exist_ok=True)
    print('%s  mAP50=%.4f  mAP50-95=%.4f' % (tag.upper(), m.box.map50, m.box.map))
    return m

mv = clean_eval('val', 'val')
CONF = 0.25
for x, y, _, yl in mv.curves_results:
    if yl == 'F1':
        px = np.asarray(x); y = np.asarray(y); f1 = y.mean(0) if y.ndim > 1 else y
        CONF = round(float(px[f1.argmax()]), 3)
print('operating conf (val F1) =', CONF)
for split, tag in (('val', 'val'), ('test', 'test')):
    t = best.val(data=DATA, split=split, conf=CONF, plots=True, project=RUNS, name=f'run5_clean_{tag}_tmp', exist_ok=True)
    for f in ('confusion_matrix.png', 'confusion_matrix_normalized.png'):
        shutil.copy(f'{RUNS}/run5_clean_{tag}_tmp/{f}', f'{RUNS}/run5_clean_{tag}/{f}')
    shutil.rmtree(f'{RUNS}/run5_clean_{tag}_tmp')

mt = clean_eval('test', 'test')

# pooled clean val+test
POOL = '/content/iadd_pool'
os.system(f'rm -rf {POOL}'); os.makedirs(f'{POOL}/images/pool'); os.makedirs(f'{POOL}/labels/pool')
for sp in ('val', 'test'):
    for img in glob.glob(f'/content/iadd_subset_v4/images/{sp}/*.jpg'):
        b = os.path.basename(img); os.symlink(img, f'{POOL}/images/pool/{b}')
        lb = f'/content/iadd_subset_v4/labels/{sp}/' + b[:-4] + '.txt'
        if os.path.exists(lb): os.symlink(lb, f'{POOL}/labels/pool/' + b[:-4] + '.txt')
yaml.safe_dump({'path': POOL, 'val': 'images/pool', 'nc': 6,
                'names': ['person','car','motorcycle','bus','truck','traffic_light']},
               open(f'{POOL}/data.yaml', 'w'))
mp = best.val(data=f'{POOL}/data.yaml', split='val', plots=False, project=RUNS, name='run5_clean_pool', exist_ok=True)
print('\nCLEAN POOLED (val+test)  mAP50=%.4f  mAP50-95=%.4f' % (mp.box.map50, mp.box.map))

---
Compare these CLEAN numbers with the earlier leaky ones (test 0.879 / val 0.747).
- If test drops toward val -> the leakage was inflating test (good that we caught it).
- If it stays similar -> the leakage was minor and the result is genuine.
Report the CLEAN test + CLEAN pooled as the honest, leakage-free benchmark.